# 02. Preprocessing & Tokenization Pipeline
### Final Project: Multilingual Health Question Answering in Low-Resource African Languages

This notebook demonstrates our data cleaning strategies, language-specific prompt prefixing,
and evaluates tokenizer fracturing and vocabulary indexing for low-resource languages.

---


In [ ]:
import os
import sys
import pandas as pd
from transformers import AutoTokenizer

# Align paths to make imports seamless
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.data_preprocessing import clean_text, apply_language_prefix, simulate_back_translation


## 1. Text Normalization and Cleaning (EXP-02)
We test cleaning noise, stripping HTML markers, and applying Unicode NFKC normalization.


In [ ]:
raw_sample = "<p>Je, ni  dalili  gani za hatari wakati wa ujauzito?</p>"
cleaned = clean_text(raw_sample)

print("Raw Sample:")
print(raw_sample)
print("\nCleaned Sample (Standardized whitespace & unicode merged):")
print(cleaned)


## 2. Language-Specific Prompt Prefixing (EXP-03)
To guide the multilingual decoder (like `mT5`), we append source language tags to the questions.


In [ ]:
question = "Je, ni dalili gani za hatari?"
prefix_sample = apply_language_prefix(question, "Kiswahili")
print("Standard Prompt Format:")
print(prefix_sample)


## 3. Back-Translation Data Augmentation Simulation (EXP-04)
Simulating back-translation-based data expansions by modifying synonms to double data samples.


In [ ]:
orig_text = "Nifanye nini kumlinda mtoto mchanga dhidi ya malaria?"
augmented_text = simulate_back_translation(orig_text, "Kiswahili")

print("Original Text:")
print(orig_text)
print("\nAugmented Text:")
print(augmented_text)


## 4. Evaluating Tokenizer Fracturing (Sub-word Over-fragmentation)
We load the pre-trained `google/mt5-small` tokenizer. We observe how low-resource languages
are fractured compared to high-resource languages, demonstrating why max sequence lengths must be handled carefully.


In [ ]:
model_name = "hf-internal-testing/tiny-random-t5" 
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Sample phrases
kiswahili_phrase = "Je, ni dalili gani za hatari?"
english_phrase = "What are the danger signs?"

# Tokenize and view tokens
k_tokens = tokenizer.tokenize(kiswahili_phrase)
e_tokens = tokenizer.tokenize(english_phrase)

print(f"Kiswahili text: '{kiswahili_phrase}'")
print(f"Kiswahili sub-word tokens ({len(k_tokens)}): {k_tokens}")
print(f"Kiswahili tokens IDs: {tokenizer.encode(kiswahili_phrase)}")

print(f"\nEnglish text: '{english_phrase}'")
print(f"English sub-word tokens ({len(e_tokens)}): {e_tokens}")
print(f"English tokens IDs: {tokenizer.encode(english_phrase)}")


## 5. Building PyTorch Dataset Objects
Loading a sample DataFrame and building our PyTorch Dataset objects ready for PyTorch dataloaders.


In [ ]:
from src.dataset import MultilingualQADataset

df_mock = pd.DataFrame([
    {"Question": "Je, ni dalili gani za hatari?", "Answer": "Dalili za hatari ni pamoja na kuvuja damu.", "Language": "Kiswahili"},
    {"Question": "Obubonero obw'akabi kuki?", "Answer": "Obubonero obw'akabi mwe muli okutonnya omusaayi.", "Language": "Luganda"}
])

dataset = MultilingualQADataset(
    df=df_mock,
    tokenizer=tokenizer,
    max_source_length=32,
    max_target_length=32,
    use_prefix=True
)

print(f"Dataset created successfully. Size: {len(dataset)}")
sample_batch = dataset[0]
print("Keys in batch sample:", sample_batch.keys())
print("Tokens shape:", sample_batch["input_ids"].shape)
print("Labels shape:", sample_batch["labels"].shape)
